# Learned Positional Embeddings — Solution

Reference implementation for `02_learned_exercise.ipynb`.

## Setup

In [1]:
import torch
import torch.nn as nn

## Solution 1 — LearnedPositionalEmbedding

In [2]:
class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        # nn.Embedding is just a (max_len, d_model) learnable matrix + an index-lookup function.
        # BERT/GPT-2 use this idiom because it mirrors how token embeddings are stored.
        self.position_embeddings = nn.Embedding(max_len, d_model)

    def forward(self, token_embeddings):
        # token_embeddings: (batch, seq_len, d_model)
        seq_len = token_embeddings.shape[1]
        positions = torch.arange(seq_len, device=token_embeddings.device)
        pos_emb = self.position_embeddings(positions)   # (seq_len, d_model)
        return token_embeddings + pos_emb               # broadcasts over batch

**Why `nn.Embedding` over `nn.Parameter`?** Both work — under the hood, `nn.Embedding(N, D)` *is* a `(N, D)` `nn.Parameter` plus an index-lookup function. Two reasons to prefer `nn.Embedding`:

1. **Symmetry with token embeddings.** BERT and GPT-2 store both positions and tokens in `nn.Embedding` modules, so the forward becomes `token_emb(token_ids) + pos_emb(position_ids)` — clean and symmetric.
2. **Loud boundary failure.** `nn.Embedding` raises `IndexError` (or a CUDA assert) when you index past `max_len`. With raw `nn.Parameter` slicing, you'd silently get a wrong-shaped output instead. We'll see this in Solution 3.

The full alternative using `nn.Parameter` is shown in the cell below — functionally identical to the `nn.Embedding` version. Pick one for consistency.

In [3]:
class LearnedPE_Param(nn.Module):
    """Alternative implementation using nn.Parameter directly. Functionally identical to the nn.Embedding version above."""

    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.P = nn.Parameter(torch.empty(max_len, d_model))
        nn.init.normal_(self.P, std=0.02)   # BERT-style init

    def forward(self, x):
        return x + self.P[:x.shape[1]]

## Solution 2 — Gradient check

Forward, backward, and confirm gradients flow through the position table.

In [4]:
torch.manual_seed(0)
module = LearnedPositionalEmbedding(max_len=128, d_model=64)

x = torch.randn(2, 10, 64)
out = module(x)
print(f"Forward output shape: {tuple(out.shape)}")

# Backward
out.sum().backward()

# Every parameter should now have a non-None .grad with non-zero values
for name, param in module.named_parameters():
    g = param.grad
    print(f"  {name}: grad shape {tuple(g.shape)}, "
          f"mean |grad| = {g.abs().mean().item():.6f}")

Forward output shape: (2, 10, 64)
  position_embeddings.weight: grad shape (128, 64), mean |grad| = 0.156250


## Solution 3 — Boundary failure

Learned PE has no entry for positions beyond `max_len`. Demonstrate the failure mode.

In [5]:
# Build a model with max_len=512, then try to feed it a length-1000 sequence.
module = LearnedPositionalEmbedding(max_len=512, d_model=64)
x_too_long = torch.zeros(1, 1000, 64)

try:
    out = module(x_too_long)
    print(f"Unexpectedly succeeded — output shape: {tuple(out.shape)}")
except (IndexError, RuntimeError) as e:
    print(f"Failed as expected: {type(e).__name__}")
    print(f"  message: {e}")

# Contrast: sinusoidal_pe(1000, 64) (from the previous notebook) works fine —
# the formula generates encodings for any position, even past training length.

Failed as expected: IndexError
  message: index out of range in self


**The fundamental limitation.** What you just observed — `IndexError` at position 512 — is the defining weakness of absolute learned positional embeddings: **there is literally no entry for positions beyond `max_len`**, because the table never had one.

Sinusoidal handles this gracefully (the formula computes an encoding for any position), but in practice both methods fail to extrapolate well past training length — even sinusoidal — because the model only ever saw position values up to `max_len` during training. Position 5000 with sinusoidal produces a *valid* number, but the attention patterns never learned what to do with that number.

This is the entire motivation for **relative** positional methods (covered in Part 2). Instead of giving each absolute position a unique fingerprint, relative methods encode only how *far apart* two tokens are. Distance 5 means the same thing whether the tokens are at positions (3, 8) or (995, 1000), so length generalization comes much closer to working out of the box.

## Run the tests

In [6]:
from tests import run_learned_tests
run_learned_tests(LearnedPositionalEmbedding)

Running LearnedPositionalEmbedding...
  ✓ Param count == max_len * d_model
  ✓ Forward shape correct
  ✓ Gradients flow through position table
  ✓ Boundary fails for seq_len > max_len

  All 4 checks passed ✓



True